# 03 · Sentiment Models

**Goal:** compare four approaches to classifying tweet sentiment and pick one for automatic monitoring.

| Model | Type | Input |
|---|---|---|
| **VADER** | Rule-based, no training | `text_clean` |
| **Naive Bayes** | Classical ML | `text_tokens` (TF-IDF) |
| **Logistic Regression** | Classical ML | `text_tokens` (TF-IDF) |
| **DistilBERT** | Pre-trained deep learning | `text_clean` |

DistilBERT takes a few minutes on a CPU. Predictions of every model are saved for the evaluation in notebook 04.

## 1. Setup and data

In [1]:
import pandas as pd
import numpy as np
import pickle
import time
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 100)

In [2]:
df = pd.read_csv('../data/processed/tweets_clean.csv')
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head(3)

Shape: (14464, 7)

Columns: ['tweet_id', 'airline', 'sentiment', 'negative_reason', 'text', 'text_clean', 'text_tokens']


,tweet_id,airline,sentiment,negative_reason,text,text_clean,text_tokens
0,567588278875213824,JetBlue,neutral,NaN,@JetBlue's new CEO seeks the right balance to please passengers and Wall ... - Greenfield Daily ...,s new ceo seeks the right balance to please passengers and wall greenfield daily reporter,new ceo seeks right balance please passengers wall greenfield daily reporter
1,567590027375702016,JetBlue,negative,Can't Tell,@JetBlue is REALLY getting on my nerves !! 😡😡 #nothappy,is really getting on my nerves nothappy,really getting nerves nothappy
2,567591480085463040,United,negative,Late Flight,@united yes. We waited in line for almost an hour to do so. Some passengers just left not wantin...,yes we waited in line for almost an hour to do so some passengers just left not wanting to wait ...,yes waited line almost hour passengers left not wanting wait past


## 2. Train-test split

The data is split once into 80% training and 20% test data, and **every model is evaluated on the same test set**, so the comparison is fair. The split is stratified, so both sets have the same sentiment mix.

In [3]:
# stratify=y keeps the same sentiment mix in train and test
X = df[['text_clean', 'text_tokens']]
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Train: {X_train.shape[0]:,} samples")
print(f"Test:  {X_test.shape[0]:,} samples")
print("\nSentiment mix in the training set:")
print(y_train.value_counts(normalize=True).round(3))
print("\nSentiment mix in the test set:")
print(y_test.value_counts(normalize=True).round(3))

Train: 11,571 samples
Test:  2,893 samples

Sentiment mix in the training set:
sentiment
negative    0.628
neutral     0.211
positive    0.161
Name: proportion, dtype: float64

Sentiment mix in the test set:
sentiment
negative    0.628
neutral     0.211
positive    0.161
Name: proportion, dtype: float64


---

## 3. Model 1: VADER (rule-based)

**VADER** (Valence Aware Dictionary and sEntiment Reasoner) scores text with a sentiment lexicon and rules for intensity, negation and punctuation.

- **Strengths:** no training needed, very fast, designed for social media text.
- **Weaknesses:** English only, misses context and sarcasm, and cannot learn airline-specific language.

In [4]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()

# One example first
sample_text = "This flight was absolutely terrible. Worst experience ever."
scores = vader.polarity_scores(sample_text)
print(f"Text: {sample_text}")
print(f"Scores: {scores}")
print(f"Compound score: {scores['compound']}")
print("\n→ compound > 0.05: positive | < -0.05: negative | else: neutral")

Text: This flight was absolutely terrible. Worst experience ever.
Scores: {'neg': 0.564, 'neu': 0.436, 'pos': 0.0, 'compound': -0.8304}
Compound score: -0.8304

→ compound > 0.05: positive | < -0.05: negative | else: neutral


In [5]:
def vader_predict(text):
    """Convert VADER compound score → label sentiment."""
    if not isinstance(text, str) or len(text) == 0:
        return 'neutral'
    score = vader.polarity_scores(text)['compound']
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

# Predict di test set
start = time.time()
vader_preds = X_test['text_clean'].apply(vader_predict)
vader_time = time.time() - start

vader_acc = accuracy_score(y_test, vader_preds)
print(f"Time: {vader_time:.2f}s")
print(f"VADER accuracy: {vader_acc:.4f} ({vader_acc*100:.2f}%)")
print(f"\n{classification_report(y_test, vader_preds)}")

Time: 0.13s
VADER accuracy: 0.5544 (55.44%)



              precision    recall  f1-score   support

    negative       0.91      0.50      0.65      1816
     neutral       0.40      0.44      0.42       611
    positive       0.34      0.90      0.50       466

    accuracy                           0.55      2893
   macro avg       0.55      0.61      0.52      2893
weighted avg       0.71      0.55      0.58      2893



---

## 4. Model 2: Naive Bayes + TF-IDF

**TF-IDF** turns text into numbers: a word gets a high weight when it is frequent in a tweet but rare across all tweets, such as *delayed*.

**Naive Bayes** is a probabilistic classifier that assumes words are independent of each other. The assumption is simplistic, but the model is a strong, fast baseline for text classification.

In [6]:
# Step 1: Convert text -> TF-IDF vectors
tfidf = TfidfVectorizer(
    max_features=5000,    # keep the 5,000 most frequent terms
    ngram_range=(1, 2),   # single words and word pairs
    min_df=2,             # a term must appear in at least 2 tweets
)

X_train_tfidf = tfidf.fit_transform(X_train['text_tokens'])
X_test_tfidf = tfidf.transform(X_test['text_tokens'])

print(f"TF-IDF shape (train): {X_train_tfidf.shape}")
print(f"TF-IDF shape (test):  {X_test_tfidf.shape}")
print("\nFirst 30 features (alphabetical):")
print(list(tfidf.get_feature_names_out())[:30])

TF-IDF shape (train): (11571, 5000)
TF-IDF shape (test):  (2893, 5000)

First 30 features (alphabetical):
['aa', 'aa flight', 'aa not', 'aadvantage', 'abc', 'ability', 'able', 'able get', 'abq', 'absolute', 'absolute worst', 'absolutely', 'absurd', 'abt', 'abysmal', 'accept', 'acceptable', 'accepted', 'accepting', 'access', 'access journal', 'accident', 'accidentally', 'accommodate', 'accommodating', 'accommodations', 'according', 'account', 'account help', 'accountability']


In [7]:
# Step 2: Train Naive Bayes
start = time.time()
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_train_time = time.time() - start

# Predict
nb_preds = nb_model.predict(X_test_tfidf)
nb_acc = accuracy_score(y_test, nb_preds)

print(f"Training time: {nb_train_time:.2f}s")
print(f"Naive Bayes accuracy: {nb_acc:.4f} ({nb_acc*100:.2f}%)")
print(f"\n{classification_report(y_test, nb_preds)}")

Training time: 0.02s
Naive Bayes accuracy: 0.7466 (74.66%)

              precision    recall  f1-score   support

    negative       0.73      0.98      0.84      1816
     neutral       0.75      0.27      0.39       611
    positive       0.90      0.45      0.60       466

    accuracy                           0.75      2893
   macro avg       0.79      0.57      0.61      2893
weighted avg       0.76      0.75      0.71      2893



---

## 5. Model 3: Logistic Regression + TF-IDF

**Logistic Regression** is a linear model and a strong baseline for text classification. Unlike Naive Bayes it does not assume independent words, and `class_weight='balanced'` offsets the dominance of negative tweets.

In [8]:
# Train Logistic Regression
start = time.time()
lr_model = LogisticRegression(
    max_iter=1000,        # enough iterations to converge
    class_weight='balanced',  # offsets the class imbalance (negative dominates)
    random_state=42,
    n_jobs=-1,            # use all CPU cores
)
lr_model.fit(X_train_tfidf, y_train)
lr_train_time = time.time() - start

# Predict
lr_preds = lr_model.predict(X_test_tfidf)
lr_acc = accuracy_score(y_test, lr_preds)

print(f"Training time: {lr_train_time:.2f}s")
print(f"Logistic Regression accuracy: {lr_acc:.4f} ({lr_acc*100:.2f}%)")
print(f"\n{classification_report(y_test, lr_preds)}")

Training time: 0.37s
Logistic Regression accuracy: 0.7774 (77.74%)

              precision    recall  f1-score   support

    negative       0.89      0.82      0.85      1816
     neutral       0.56      0.67      0.61       611
    positive       0.71      0.75      0.73       466

    accuracy                           0.78      2893
   macro avg       0.72      0.75      0.73      2893
weighted avg       0.79      0.78      0.78      2893



### Most indicative words

Logistic Regression coefficients show which words push a tweet toward each sentiment, which makes the model easy to explain.

In [9]:
feature_names = tfidf.get_feature_names_out()
classes = lr_model.classes_

# For each class, the 15 terms with the highest coefficients
for i, class_name in enumerate(classes):
    coefs = lr_model.coef_[i]
    top_indices = coefs.argsort()[-15:][::-1]
    top_words = [(feature_names[idx], coefs[idx]) for idx in top_indices]
    
    print(f"\nTop terms for {class_name}:")
    for word, coef in top_words:
        print(f"   {word:30s} {coef:+.3f}")


Top terms for negative:
   not                            +4.674
   hours                          +3.685
   no                             +3.552
   worst                          +3.405
   delayed                        +3.215
   hour                           +3.145
   hold                           +2.936
   hrs                            +2.632
   luggage                        +2.479
   cancelled                      +2.393
   nothing                        +2.250
   lost                           +2.144
   stuck                          +2.138
   bags                           +2.129
   system                         +2.123

Top terms for neutral:
   hi                             +1.929
   policy                         +1.822
   possible                       +1.528
   number                         +1.436
   mexico                         +1.415
   chance                         +1.404
   carry                          +1.360
   no not                         +1.351
   dm   

---

## 6. Model 4: DistilBERT (pre-trained transformer)

**DistilBERT** is a smaller, faster version of BERT. This notebook uses the version fine-tuned on SST-2 (movie review sentiment) without retraining it on airline tweets.

- It only predicts **positive or negative**. Predictions below a confidence threshold of 0.7 are treated as **neutral**.
- It runs on a GPU if one is available; on a CPU the test set takes about a minute.

In [10]:
import torch
from huggingface_hub.utils import logging as hub_logging
from transformers import pipeline
from transformers.utils import logging as hf_logging

# Keep download and loading progress out of the notebook output
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
hub_logging.set_verbosity_error()

# Use a GPU when available
device = 0 if torch.cuda.is_available() else -1
device_name = "GPU" if device == 0 else "CPU"
print(f"Running on: {device_name}")

# Load pre-trained sentiment pipeline
print("Loading DistilBERT (about 270 MB on the first run)")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
    truncation=True,
    max_length=128,
)

Running on: CPU
Loading DistilBERT (about 270 MB on the first run)


In [11]:
def hf_predict_with_neutral(text, threshold=0.7):
    """
    DistilBERT (SST-2) only predicts positive or negative.
    Predictions below the confidence threshold are treated as neutral.
    """
    if not isinstance(text, str) or len(text.strip()) == 0:
        return 'neutral'
    try:
        result = sentiment_pipeline(text[:512])[0]  # cap the input length
        label = result['label'].lower()  # 'positive' atau 'negative'
        score = result['score']
        
        # Low confidence -> neutral
        if score < threshold:
            return 'neutral'
        return label
    except Exception:
        return 'neutral'

# A few sanity checks
test_examples = [
    "This flight was amazing! Best service ever.",
    "Worst flight ever, never flying with them again.",
    "Flight took off at 3pm from JFK.",
]
for t in test_examples:
    print(f"'{t}' → {hf_predict_with_neutral(t)}")

'This flight was amazing! Best service ever.' → positive
'Worst flight ever, never flying with them again.' → negative
'Flight took off at 3pm from JFK.' → positive


In [12]:
# Predict the whole test set, one tweet at a time

start = time.time()
hf_preds = X_test['text_clean'].apply(hf_predict_with_neutral)
hf_time = time.time() - start

hf_acc = accuracy_score(y_test, hf_preds)
print(f"\nInference time: {hf_time:.1f}s ({hf_time/60:.1f} min)")
print(f"DistilBERT accuracy: {hf_acc:.4f} ({hf_acc*100:.2f}%)")
print(f"\n{classification_report(y_test, hf_preds)}")


Inference time: 53.5s (0.9 min)
DistilBERT accuracy: 0.7093 (70.93%)

              precision    recall  f1-score   support

    negative       0.79      0.90      0.84      1816
     neutral       0.32      0.04      0.06       611
    positive       0.53      0.85      0.65       466

    accuracy                           0.71      2893
   macro avg       0.55      0.59      0.52      2893
weighted avg       0.65      0.71      0.65      2893



---

## 7. Comparison

In [13]:
results = pd.DataFrame({
    'Model': ['VADER', 'Naive Bayes', 'Logistic Regression', 'DistilBERT'],
    'Type': ['Rule-based', 'Classical ML', 'Classical ML', 'Deep Learning'],
    'Accuracy': [vader_acc, nb_acc, lr_acc, hf_acc],
    'Time (s)': [vader_time, nb_train_time, lr_train_time, hf_time],
}).sort_values('Accuracy', ascending=False).reset_index(drop=True)

results['Accuracy %'] = (results['Accuracy'] * 100).round(2)
print("Accuracy on the test set:")
print(results[['Model', 'Type', 'Accuracy %', 'Time (s)']].to_string(index=False))

Accuracy on the test set:
              Model          Type  Accuracy %  Time (s)
Logistic Regression  Classical ML       77.74  0.365891
        Naive Bayes  Classical ML       74.66  0.019359
         DistilBERT Deep Learning       70.93 53.494542
              VADER    Rule-based       55.44  0.128632


## 8. Save predictions and models

Predictions go to notebook 04 for evaluation. The TF-IDF vectorizer and the two classical models are saved for the dashboard.

In [14]:
# All predictions in one table, for notebook 04
predictions = pd.DataFrame({
    'text_clean': X_test['text_clean'].values,
    'text_tokens': X_test['text_tokens'].values,
    'true_label': y_test.values,
    'vader_pred': vader_preds.values,
    'nb_pred': nb_preds,
    'lr_pred': lr_preds,
    'hf_pred': hf_preds.values,
})

predictions.to_csv('../data/processed/model_predictions.csv', index=False)
print(f"Saved predictions: {predictions.shape}")
predictions.head()

Saved predictions: (2893, 7)


,text_clean,text_tokens,true_label,vader_pred,nb_pred,lr_pred,hf_pred
0,so no idea thanks,no idea thanks,neutral,negative,negative,neutral,negative
1,does anyone realize that banning peanuts on a flight due to an allergic person ignores p nut oil...,anyone realize banning peanuts flight due allergic person ignores nut oil dust previous flights,negative,negative,negative,negative,negative
2,this was the rd cancellation in days due to weather in nashville trips to the airport each time ...,rd cancellation days due weather nashville trips airport time communicated late sw,negative,neutral,negative,negative,negative
3,oh my gosh going to dm you now thank you,oh gosh going dm thank,positive,positive,negative,neutral,positive
4,i just wish that a team member would pick up the phone at this point let alone be courteous or h...,wish team member would pick phone point let alone courteous helpful,negative,positive,negative,positive,negative


In [15]:
# Save model objects (TF-IDF + classical ML models)
import os
os.makedirs('../models', exist_ok=True)

with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("Saved models/tfidf_vectorizer.pkl")

with open('../models/naive_bayes.pkl', 'wb') as f:
    pickle.dump(nb_model, f)
print("Saved models/naive_bayes.pkl")

with open('../models/logistic_regression.pkl', 'wb') as f:
    pickle.dump(lr_model, f)
print("Saved models/logistic_regression.pkl")

# Save results comparison
results.to_csv('../data/processed/model_results.csv', index=False)
print("Saved data/processed/model_results.csv")

Saved models/tfidf_vectorizer.pkl
Saved models/naive_bayes.pkl
Saved models/logistic_regression.pkl
Saved data/processed/model_results.csv


## 9. Conclusion

| Rank | Model | Accuracy |
|---|---|---|
| 1 | **Logistic Regression** | **77.7%** |
| 2 | Naive Bayes | 74.7% |
| 3 | DistilBERT | 70.9% |
| 4 | VADER | 55.4% |

- **Logistic Regression is the most accurate** and trains in under a second.
- **A pre-trained transformer is not automatically better.** DistilBERT was trained on movie reviews and only knows positive and negative, so it struggles with neutral tweets. Fine-tuning it on airline tweets would likely close the gap, at a much higher cost.
- **VADER is the fastest but least accurate**: a general-purpose lexicon misses airline-specific complaints such as *hold* or *hours*.

**Next, notebook 04:** confusion matrices, per-class scores and error analysis, to choose the model for the dashboard.